# Main Figure 6

Frozen drawing logic and saved derived inputs. Outputs: `outputs/main/figure_06`. See reproduction status before release.


In [ ]:
from pathlib import Path
import os, sys
sys.dont_write_bytecode = True
_start = Path.cwd().resolve()
REPO = next((p for q in [_start, *_start.parents] for p in [q, q / "github"]
             if (p / "plot_utils" / "release_runtime.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Run from the repository root or a notebook subdirectory.")
sys.path.insert(0, str(REPO / "plot_utils"))
from release_runtime import configure
configure(REPO)


In [ ]:
from pathlib import Path
import os,sys,json,hashlib
PROJECT_ROOT=ROOT=REPO/'figure_source_data/shared'
OUT=REPO/'outputs/main/figure_06'; OUT.mkdir(parents=True,exist_ok=True); RT=OUT/'runtime'; RT.mkdir(exist_ok=True)
DATA=ROOT/'new_fig_try/fig6/calculated_results/v3'
for k in ['MPLCONFIGDIR','TEMP','TMP','TMPDIR','XDG_CACHE_HOME']: os.environ[k]=str(RT)
sys.dont_write_bytecode=True
import numpy as np
import pandas as pd
from scipy.stats import theilslopes
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
SRC=ROOT/'Part8_change_work'
P=['GRUN','ERA5','GLDAS']; COL=dict(zip(P,['#007C78','#CB7935','#667BA5']))
YEARS=np.arange(1986,2014); THRESHOLDS=[.55,.575,.60,.625,.65]
reads={}
def read_csv(p,check=None):
    h=hashlib.sha256(p.read_bytes()).hexdigest()
    reads[str(p)]=h
    return pd.read_csv(p)
annual=read_csv(SRC/'tables/corrected_annual.csv')
decile=read_csv(SRC/'tables/corrected_decile.csv')
trends=read_csv(SRC/'tables/corrected_trends.csv')
concord=read_csv(SRC/'tables/concordance_full_precision.csv')
pooled=read_csv(SRC/'tables/corrected_pooled.csv')
retro=read_csv(SRC/'figure_source_data/Figure5_source_data_deciles.csv')



plt.rcParams['font.family']='sans-serif'
plt.rcParams['font.sans-serif']=['Arial','DejaVu Sans','Liberation Sans']
plt.rcParams['svg.fonttype']='none'


In [ ]:
heat=read_csv(DATA/'panel_a_pooled_auc.csv')
annual_auc=read_csv(DATA/'annual_high_severity_auc.csv')
sensitivity=read_csv(DATA/'panel_e_threshold_trends.csv')
fixed=read_csv(DATA/'panel_f_fixed_lead_trends.csv')
decile=read_csv(DATA/'panel_b_Lstar.csv')
annual=read_csv(DATA/'panel_c_annual_high.csv')
concord=read_csv(DATA/'panel_d_contrasts.csv')
retro=read_csv(DATA/'panel_d_retrospective_components.csv')
trends=read_csv(DATA/'panel_d_prospective_components.csv')


In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.titlesize':11,'axes.labelsize':9,'xtick.labelsize':8,'ytick.labelsize':8,'axes.linewidth':.65,'pdf.fonttype':42,'svg.fonttype':'none'})
fig=plt.figure(figsize=(7.20472440945,180/25.4))
grid=fig.add_gridspec(2,3,left=.075,right=.97,bottom=.14,top=.90,wspace=.49,hspace=.62)
top=[grid[0,0],grid[0,1],grid[0,2]]
bottom=[grid[1,0],grid[1,1],grid[1,2]]

def label(spec,letter,title):
    pos=spec.get_position(fig)
    fig.text(pos.x0-.026,pos.y1+.044,letter,fontweight='bold',fontsize=14)
    fig.text(pos.x0,pos.y1+.044,title,fontsize=11)
def clean(ax):ax.spines[['top','right']].set_visible(False)
label(top[0],'a','Prospective predictability landscape')
sub=top[0].subgridspec(1,3,wspace=.12)
for i,p in enumerate(P):
    ax=fig.add_subplot(sub[i]); z=heat.query('product==@p').pivot(index='decile',columns='lead',values='auc')
    im=ax.imshow(z,origin='lower',extent=(.5,12.5,.5,10.5),aspect='auto',vmin=.5,vmax=.70,cmap='YlGnBu',interpolation='nearest')
    if z.to_numpy().min()<.6<z.to_numpy().max():ax.contour(np.arange(1,13),np.arange(1,11),z,levels=[.6],colors=['white'],linewidths=1)
    ax.set(xticks=[1,6,12],yticks=[1,5,10]);ax.set_title(p,color=COL[p],fontsize=10,pad=7)
    if i==0:ax.set_ylabel('Severity decile')
    else:ax.set_yticklabels([])
    if i==1:ax.set_xlabel('Forecast lead L (months)')
pos=top[0].get_position(fig); cbax=fig.add_axes([pos.x0,pos.y0-.034,pos.width,.006]);cb=fig.colorbar(im,cax=cbax,orientation='horizontal',ticks=[.50,.55,.60,.65,.70]);cb.set_label('Pooled AUC · white boundary: 0.60',fontsize=8,labelpad=2);cb.outline.set_linewidth(.5)
label(top[1],'b','Usable lead across severity')
sub=top[1].subgridspec(3,1,hspace=.22)
for i,p in enumerate(P):
    ax=fig.add_subplot(sub[i]);z=decile.query('product==@p')
    ax.fill_between(z.decile,0,z.Lstar,color=COL[p],alpha=.09,lw=0)
    ax.fill_between(z.decile,z.ci_low,z.ci_high,color=COL[p],alpha=.3,lw=0)
    ax.plot(z.decile,z.Lstar,color=COL[p],lw=1.3);ax.scatter(z.decile,z.Lstar,s=9,color=COL[p],zorder=3)
    ax.set(xlim=(.8,10.2),ylim=(-.6,12.6),yticks=[0,6,12],xticks=[1,5,10]);clean(ax)
    ax.text(.02,.78,p,transform=ax.transAxes,color=COL[p],fontsize=8,fontweight='bold')
    if p=='GRUN':ax.text(.39,.30,'threshold floor',transform=ax.transAxes,color=COL[p],fontsize=8)
    if i<2:ax.set_xticklabels([])
    else:ax.set_xlabel('Severity decile')
    if i==1:ax.set_ylabel('L* (months)')
label(top[2],'c','High-severity usable lead through time')
sub=top[2].subgridspec(2,1,height_ratios=[4,1],hspace=.32)
ax=fig.add_subplot(sub[0]);clean(ax)
for p in ['ERA5','GLDAS']:
    z=annual.query("product==@p and group=='HIGH'").sort_values('year')
    ax.plot(z.year,z.Lstar,'o-',color=COL[p],lw=1.3,ms=3,label=p)
ax.set(ylim=(0,12),xlim=(1985.5,2013.5),ylabel='L* (months)',xticks=[1986,1995,2004,2013]);ax.set_xticklabels([])
ax.legend(frameon=False,fontsize=8,ncol=2,loc='upper left')
ax=fig.add_subplot(sub[1]);z=annual.query("product=='GRUN' and group=='HIGH'").sort_values('year')
ax.bar(z.year,z.Lstar,width=.8,color=COL['GRUN']);ax.scatter(z.year[z.Lstar==0],z.Lstar[z.Lstar==0],marker='|',s=25,color=COL['GRUN'],zorder=3)
ax.set(xlim=(1985.5,2013.5),ylim=(-.3,12),yticks=[0,12],xticks=[1986,1995,2004,2013],xlabel='Evaluation year');clean(ax)
ax.text(.015,.80,'GRUN',transform=ax.transAxes,color=COL['GRUN'],fontsize=8)
label(bottom[0],'d','Paired retrospective–prospective synthesis')
sub=bottom[0].subgridspec(1,2,wspace=.33)
for j,title in enumerate(['Retrospective PT','Prospective L*']):
    ax=fig.add_subplot(sub[j]);clean(ax);ax.axvline(0,color='.7',ls='--',lw=.7)
    for i,p in enumerate(P):
        r=concord.query('product==@p').iloc[0]
        if j==0:
            q=retro.query('product==@p').set_index('severity_decile');lo=q.loc[1];hi=q.loc[10]
            low,high=lo.slope,hi.slope;lc=[lo.CI_low,lo.CI_high];hc=[hi.CI_low,hi.CI_high];delta=r.retro_gradient
        else:
            q=trends.query("product==@p and metric=='Lstar'").set_index('group');lo=q.loc['LOW'];hi=q.loc['HIGH']
            low,high=lo.value,hi.value;lc=[lo.ci_low,lo.ci_high];hc=[hi.ci_low,hi.ci_high];delta=r.prospective_gradient
        assert abs(high-low-delta)<1e-12
        ax.plot([low,high],[i-.09,i+.09],color=COL[p],lw=1.3,alpha=.75)
        for v,ci,y,filled in [(low,lc,i-.09,False),(high,hc,i+.09,True)]:
            ax.plot(ci,[y,y],color=COL[p],lw=.85,alpha=.6)
            ax.plot(v,y,'o',ms=5,mfc=COL[p] if filled else 'white',mec=COL[p],mew=1,zorder=3)
        ax.text(.97,i-.37,f'Δ = {delta:+.3f}',transform=ax.get_yaxis_transform(),ha='right',fontsize=8,color=COL[p],bbox={'facecolor':'white','edgecolor':'none','pad':.5})
    ax.set(ylim=(2.6,-.6),yticks=[0,1,2],yticklabels=P if j==0 else [],xlabel='Sen slope (months decade⁻¹)')
    ax.set_title(title,fontsize=9,pad=8)
    ax.set_xlim((-.10,.065) if j==0 else (-.52,.88))
    if j==0:ax.set_xticks([-.08,-.04,0,.04])
    else:ax.set_xticks([-.4,0,.4,.8])
pos=bottom[0].get_position(fig)
fig.legend([Line2D([],[],marker='o',mfc='white',color='.35',ls='none'),Line2D([],[],marker='o',color='.35',ls='none')],['Low','High'],loc='upper left',bbox_to_anchor=(pos.x0,pos.y0-.043),frameon=False,ncol=2,fontsize=8,borderaxespad=0)
fig.text(pos.x0,pos.y0-.065,'Δ = high − low; bars: component 95% CIs',fontsize=7.5,color='.35')
label(bottom[1],'e','Threshold dependence of L* trends')
ax=fig.add_subplot(bottom[1]);clean(ax)
for p in P:
    z=sensitivity.query('product==@p');ax.plot(z.threshold,z.slope,'o-',color=COL[p],ms=4,lw=1.6,label=p)
ax.axhline(0,color='.65',lw=.8,ls='--')
ax.set(xlabel='AUC threshold',ylabel='High-severity L* trend\n(months decade⁻¹)',xticks=THRESHOLDS)
ax.tick_params(axis='x',labelsize=7);ax.legend(frameon=False,fontsize=8,loc='best')
label(bottom[2],'f','Lead-dependent discrimination trends')
sub=bottom[2].subgridspec(3,1,hspace=.24)
span=fixed.ci_high.max()-fixed.ci_low.min()
ylimits=(fixed.ci_low.min()-.10*span,fixed.ci_high.max()+.10*span)
for i,p in enumerate(P):
    ax=fig.add_subplot(sub[i]);clean(ax)
    z=fixed.query('product==@p').sort_values('lead')
    ax.axhline(0,color='.65',lw=.65,ls='--')
    ax.vlines(z.lead,z.ci_low,z.ci_high,color=COL[p],lw=.7,alpha=.5)
    ax.plot(z.lead,z.slope,color=COL[p],lw=1)
    ax.scatter(z.lead,z.slope,s=11,color=COL[p],zorder=3)
    ax.set(xlim=(.7,12.3),ylim=ylimits,xticks=[1,3,6,9,12],yticks=[0,.02])
    ax.text(.025,.77,p,transform=ax.transAxes,color=COL[p],fontsize=8,fontweight='bold')
    if i<2:ax.set_xticklabels([])
    else:ax.set_xlabel('Forecast lead L (months)')
    if i==1:ax.set_ylabel('AUC trend (decade⁻¹)')
pos=bottom[2].get_position(fig)
fig.text(pos.x0,pos.y0-.034,'Points: temporal Sen slopes; bars: 95% Sen intervals',fontsize=7.5,color='.35')
fig.text(.06,.012,'Pseudo-prospective evaluation, 1986–2013; training-only scaling',fontsize=8,color='.35')


In [ ]:

FIGURE_NUMBER=6
FROZEN_TITLES={'a': 'Discrimination varies across products and leads', 'b': 'Threshold-defined leads vary with severity', 'c': 'High-severity usable leads vary through time', 'd': 'Retrospective and prospective contrasts differ', 'e': 'High-severity trends depend on threshold', 'f': 'Discrimination trends vary with forecast lead'}
from matplotlib.text import Text
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import textwrap
OUTDIR=Path(OUT if FIGURE_NUMBER==6 else OUTPUT_ROOT)
QA=OUTDIR/'qa';QA.mkdir(exist_ok=True)
def numerical_artists():
    result={}
    def add(key,value):
        a=np.ma.asarray(value)
        if a.dtype.kind in 'biufc':
            v=np.ascontiguousarray(a.filled(np.nan) if np.ma.isMaskedArray(a) and a.dtype.kind in 'fc' else a)
            result[key]={'shape':list(v.shape),'sha256':hashlib.sha256(v.tobytes()).hexdigest()}
    for j,a in enumerate(fig.axes):
        for k,l in enumerate(a.lines):add(f'{j}/line{k}/x',l.get_xdata());add(f'{j}/line{k}/y',l.get_ydata())
        for k,l in enumerate(a.collections):
            if l.get_array() is not None:add(f'{j}/collection{k}/array',l.get_array())
            if hasattr(l,'get_offsets'):add(f'{j}/collection{k}/offsets',l.get_offsets())
            if hasattr(l,'get_segments'):
                for h,v in enumerate(l.get_segments()):add(f'{j}/collection{k}/segment{h}',v)
            if l.__class__.__name__ in ('PolyCollection','FillBetweenPolyCollection'):
                for h,v in enumerate(l.get_paths()):add(f'{j}/collection{k}/path{h}',v.vertices)
        for k,l in enumerate(a.images):add(f'{j}/image{k}',l.get_array())
        for k,l in enumerate(a.patches):
            if isinstance(l,matplotlib.patches.Rectangle):add(f'{j}/bar{k}',[l.get_x(),l.get_y(),l.get_width(),l.get_height()])
    return result
fig.canvas.draw()
before=numerical_artists()
contracts=json.loads((REPO/'assets/other/panel_contracts.json').read_text(encoding='utf-8'))
oldtitles={r[2] for r in contracts if r[0]==FIGURE_NUMBER}
notes=[]
for t in list(fig.findobj(Text)):
    s=t.get_text()
    if s in oldtitles or s in list('abcdef'):
        t.set_text('');continue
    if len(s)>180:
        notes.append(s);t.set_text('');continue
    s=s.replace('decade$^{-1}$','decade⁻¹').replace(r'$\beta$','β').replace('$R^2$','R²')
    t.set_text(s);t.set_fontfamily('DejaVu Sans');t.set_fontsize(5.5 if FIGURE_NUMBER in (1,2) else 6)
    if t.get_rotation():t.set_rotation_mode('anchor')
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial','Liberation Sans'],
    'font.size':6,'axes.labelsize':6,'xtick.labelsize':5.5,'ytick.labelsize':5.5,
    'legend.fontsize':5.5,'pdf.fonttype':42,'svg.fonttype':'none'})
DIV=plt.get_cmap('RdBu_r')
for ax in fig.axes:
    cb=getattr(ax,'_colorbar',None)
    for sp in ax.spines.values():sp.set_linewidth(.45);sp.set_edgecolor('#657076')
    ax.tick_params(width=.45,length=2,pad=2,labelsize=5.5)
    ax.xaxis.label.set_fontsize(6);ax.yaxis.label.set_fontsize(6)
    ax.title.set_fontsize(6)
    if cb is not None:
        cb.outline.set_linewidth(.35)
        ax.tick_params(length=1.5,pad=1)
        ax.set_axes_locator(None)
        p=ax.get_position()
        if cb.orientation=='horizontal':ax.set_position([p.x0,p.y0,p.width,min(p.height,2.5/(72*fig.get_figheight()))])
        else:ax.set_position([p.x0,p.y0,min(p.width,2.5/(72*fig.get_figwidth())),p.height])
    for artist in list(ax.images)+list(ax.collections):
        norm=getattr(artist,'norm',None)
        if norm is not None and norm.vmin is not None and norm.vmax is not None and norm.vmin<0<norm.vmax:
            assert np.isclose(abs(norm.vmin),abs(norm.vmax)), 'Original diverging limits require review'
            artist.set_cmap(DIV);artist.set_norm(TwoSlopeNorm(vmin=norm.vmin,vcenter=0,vmax=norm.vmax))
    for line in ax.lines:
        if line.get_linestyle() in ('--',':'):
            x=np.asarray(line.get_xdata());y=np.asarray(line.get_ydata())
            if (len(x)>0 and np.all(x==0)) or (len(y)>0 and np.all(y==0)):
                line.set_color('#8B9298');line.set_linewidth(.5);line.set_linestyle('--')
        if line.get_markersize()>4:line.set_markersize(3.5)
        if line.get_linewidth()>1.8:line.set_linewidth(1.5)
    if hasattr(ax,'projection'):
        for col in ax.collections:
            if col.__class__.__name__=='FeatureArtist':col.set_linewidth(.25)
        ax.spines['geo'].set_linewidth(.45)

if FIGURE_NUMBER==1:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='24 mo':t.set_x(.62);t.set_transform(ax.transAxes);t.set_y(.07);t.set_ha('right')
            if t.get_text().startswith('d1  '):t.set_position((0,1.02));t.set_va('bottom')
            if t.get_text().startswith('Common 36-month'):
                notes.append(t.get_text());t.set_text('Common 36-month\ncoverage cohort');t.set_position((.04,.06));t.set_fontsize(5)
if FIGURE_NUMBER==2:
    for ax in fig.axes:
        if ax.get_xlabel()=='Months relative to meteorological anchor':ax.set_xlabel('Months from meteorological anchor',fontsize=6)
        for t in ax.texts:
            if t.get_text().startswith('Negative ='):notes.append(t.get_text());t.set_text('')
            if '95% block CI' in t.get_text():t.set_position((.03,1.02));t.set_va('bottom');t.set_fontsize(5)
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb is not None and 'hexagon' in ax.get_xlabel():
            from matplotlib.ticker import FixedLocator,FixedFormatter,NullLocator
            ax.xaxis.set_major_locator(FixedLocator([1,10,100,1000]));ax.xaxis.set_major_formatter(FixedFormatter(['1','10','100','1000']));ax.xaxis.set_minor_locator(NullLocator())
if FIGURE_NUMBER==3:
    for leg in list(fig.legends):
        if any(t.get_text()=='Q10–Q90' for t in leg.get_texts()):
            handles=leg.legend_handles;labels=[t.get_text() for t in leg.get_texts()]
            pos=gs[0,0].get_position(fig);leg.remove()
            fig.legend(handles,labels,ncol=4,frameon=False,fontsize=5.2,loc='upper left',bbox_to_anchor=(pos.x0-.005,pos.y0-.038),handlelength=1.2,columnspacing=.8)
    maps=[a for a in fig.axes if hasattr(a,'projection')]
    for a in maps:
        a.title.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Main widths pool events'):
            t.set_text('Main widths: pooled events. d2/f: grid-year means. Grey: no estimate.');t.set_fontsize(5.5)
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text().startswith('Thin: annual'):
                t.set_position((.02,1.01));t.set_va('bottom');t.set_text('Thin: annual; thick: 5-year visual mean');t.set_fontsize(5)
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='horizontal':
            p=ax.get_position();ax.set_position([p.x0,p.y0-.01,p.width,p.height]);cb.set_ticks([cb.norm.vmin,0,cb.norm.vmax] if cb.norm.vmin<0 else [0,50,100])
        if ax.images:
            p=ax.get_position();ax.set_position([p.x0+.04,p.y0,p.width-.04,p.height]);ax.tick_params(axis='y',labelsize=5)
if FIGURE_NUMBER==4:
    for leg in fig.legends:leg.set_bbox_to_anchor((.5,.045))
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text().replace('Parent number','Antecedent count').replace('Many-to-one','Multiple antecedents')
            if '  ρ: ' in s:
                s=s.replace('  ρ: ','\nρ: ');t.set_position((.01,1.03));t.set_va('bottom');t.set_fontsize(5.2)
            if s.startswith('a1  '):
                t.set_text('');p=ax.get_position();fig.text(p.x0,p.y1+.013,'a1  Event scale · GRUN',fontsize=5.5,fontweight='bold')
            else:t.set_text(s)
            if not ax.axison:t.set_fontsize(5.2)
        if ax.get_ylabel()=='TP slope':ax.set_ylabel('PT slope',fontsize=6)
        cb=getattr(ax,'_colorbar',None)
        if cb is not None:cb.set_ticks([-1,0,1])
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text()
            if s.startswith('Envelope spans') or s.startswith('Filled:'):
                t.set_position((0,-.22));t.set_ha('left');t.set_va('top');t.set_fontsize(5.2)
            if s=='Severity decile':t.set_x(-.29);t.set_fontsize(5.5)
            if 'median β < 0' in s:
                t.set_position((.0,-.32));t.set_ha('left');t.set_va('top');t.set_text(s.replace('\n','; ',1));t.set_fontsize(5.2);t.set_bbox(None)
        if hasattr(ax,'projection'):
            leg=ax.get_legend()
            if leg:leg.set_bbox_to_anchor((0,-.10));leg._loc=2
            for cax in fig.axes:
                cb=getattr(cax,'_colorbar',None)
                if cb and cb.orientation=='horizontal':
                    p=ax.get_position();cax.set_position([p.x0,p.y0-.105,p.width,2.5/(72*fig.get_figheight())])
if FIGURE_NUMBER==6:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='threshold floor':t.set_text('AUC < 0.60: L* = 0');t.set_x(.34);t.set_fontsize(5.5)
            if t.get_text().startswith('Δ ='):
                t.set_x(1.0);t.set_fontsize(5);t.set_bbox(None)
                if ax.get_title()=='Retrospective PT':
                    t.set_x(.02);t.set_ha('left');t.set_text(t.get_text().replace('Δ = ','Δ '))
            if t.get_text() in ('GRUN','ERA5','GLDAS') and ax in fig.axes[12:]:
                t.set_position((.50,.84));t.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Points: temporal'):t.set_y(.025);t.set_fontsize(5)


for ax in fig.axes:
    leg=ax.get_legend()
    if leg:
        for t in leg.get_texts():t.set_fontsize(5)
    for t in ax.texts:
        if len(t.get_text())>36:t.set_text(textwrap.fill(t.get_text(),36,break_long_words=False));t.set_fontsize(5)
    if len(ax.get_xlabel())>32:ax.set_xlabel(textwrap.fill(ax.get_xlabel(),32,break_long_words=False),fontsize=5.5)
if FIGURE_NUMBER==3:
    for t in fig.texts:
        if t.get_text().startswith(('f1 ','f2 ','Bands:')):t.set_text(textwrap.fill(t.get_text(),34,break_long_words=False));t.set_fontsize(5)
    for leg in fig.legends:
        if any(t.get_text()=='Q10–Q90' for t in leg.get_texts()):
            h=leg.legend_handles;l=[t.get_text() for t in leg.get_texts()];p=gs[0,0].get_position(fig);leg.remove();fig.legend(h,l,ncol=2,frameon=False,fontsize=5,loc='upper left',bbox_to_anchor=(p.x0,p.y0-.04))
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='vertical':
            p=ax.get_position();ax.set_position([.94,p.y0,p.width,p.height]);ax.yaxis.label.set_fontsize(5)
        for t in ax.texts:
            if t.get_text()=='Severity decile':t.set_x(-.40)
        if hasattr(ax,'projection'):
            leg=ax.get_legend()
            if leg:leg.set_bbox_to_anchor((0,-.10));leg._ncols=1
            for t in ax.texts:
                if 'median β' in t.get_text():t.set_y(-.45)
if FIGURE_NUMBER==6:
    for ax in fig.axes:
        if ax.get_xlabel().startswith('Sen slope'):ax.set_xlabel('Sen slope\n(months decade⁻¹)',fontsize=5)
        cb=getattr(ax,'_colorbar',None)
        if cb:
            p=ax.get_position();ax.set_position([p.x0,p.y0-.016,p.width,p.height])
        if ax.get_title()=='Retrospective PT':ax.set_xticks([-.08,0,.04])
    for leg in fig.legends:leg.set_bbox_to_anchor((bottom[0].get_position(fig).x0,.078))
    for t in fig.texts:
        if t.get_text().startswith('Δ = high'):t.set_y(.046);t.set_text('Δ = high − low; bars:\ncomponent 95% CIs');t.set_fontsize(5)
        if t.get_text().startswith('Points: temporal'):t.set_text('Points: temporal Sen slopes;\nbars: 95% Sen intervals');t.set_y(.035);t.set_fontsize(5)


if FIGURE_NUMBER==3:
    for ax in fig.axes:
        if ax.get_title().startswith(('f1 ','f2 ')):ax.set_title(textwrap.fill(ax.get_title(),29),loc='left',fontsize=5,pad=8)
    for t in fig.findobj(Text):
        if t.get_text().startswith(('f1 ','f2 ')):t.set_text(textwrap.fill(t.get_text(),29));t.set_fontsize(5)
        if t.get_text().startswith('Bands:'):t.set_y(t.get_position()[1]-.027)
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='vertical':
            p=ax.get_position();ax.set_position([.923,p.y0,p.width,p.height]);ax.yaxis.labelpad=1
        leg=ax.get_legend()
        if leg and any(t.get_text() in ('Q50','Min. events') for t in leg.get_texts()):leg.set_bbox_to_anchor((0,-.18));leg._loc=2
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text().startswith('Filled:'):t.set_y(-.34)
if FIGURE_NUMBER==6:
    for leg in fig.legends:leg.set_bbox_to_anchor((bottom[0].get_position(fig).x0,.095))

if FIGURE_NUMBER in (1,2,3):specs=[gs[0,0],gs[0,1],gs[0,2],gs[1,0],gs[1,1],gs[1,2]]
elif FIGURE_NUMBER==4:specs=[gs[0,0],gs[0,1],gs[1,0],gs[1,1]]
elif FIGURE_NUMBER==5:specs=[outer[0,0],outer[0,1],outer[0,2],outer[1,0],outer[1,1],outer[1,2]]
else:specs=top+bottom
headers=[]
for letter,spec in zip('abcdef',specs):
    b=spec.get_position(fig);h=fig.get_figheight()*72;w=fig.get_figwidth()*72
    width_chars=33 if FIGURE_NUMBER in (1,2) else 28
    title_offset=9 if FIGURE_NUMBER in (1,2) else 24 if FIGURE_NUMBER==4 else 18
    txt=textwrap.fill(FROZEN_TITLES[letter],width_chars,break_long_words=False)
    t=fig.text(b.x0,b.y1+title_offset/h,txt,ha='left',va='bottom',fontsize=6.5,fontweight='normal',linespacing=1.4,color='#20272B')
    fig.text(b.x0-10/w,b.y1+title_offset/h,letter,ha='left',va='bottom',fontsize=8,fontweight='bold',color='#20272B')
    headers.append(t)
fig.canvas.draw()
after=numerical_artists()
assert before==after,'A scientific artist changed during styling'
(QA/'numerical_artist_invariance.json').write_text(json.dumps({'status':'PASS','entries':len(before),'before':before,'after':after},indent=2),encoding='utf-8')
(QA/'retained_caption_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
from audit_panel_alignment import require_matplotlib_panel_alignment
eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None and not hasattr(a,'projection')]
if len(eligible)<2:eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None]
ids=[f'plot{i+1}' for i in range(len(eligible))]
rows={};cols={}
for ident,ax in zip(ids,eligible):
    p=ax.get_position();rows.setdefault((round(p.y0,5),round(p.y1,5)),[]).append(ident);cols.setdefault((round(p.x0,5),round(p.x1,5)),[]).append(ident)
rg=[v for v in rows.values() if len(v)>1];cg=[v for v in cols.values() if len(v)>1]
ex=[]
for group in cg:
    if len(group)>=3:
        parent_ids={str(eligible[ids.index(k)].get_subplotspec().get_gridspec()) for k in group}
        ex.append({'panels':group,'checks':['vertical-gutter'],'reason':'Column spans distinct main-panel rows and nested stacks; outer and inner gutters intentionally differ. Stack spacing remains uniform within each original nested GridSpec.'})
for group in rg:
    if len(group)>=3:
        widths=[eligible[ids.index(k)].get_position().width for k in group]
        if np.ptp(widths)>1e-4:
            ex.append({'panels':group,'checks':['panel-width','horizontal-gutter'],'reason':'Nested composites intentionally allocate widths to distinct quantities; within-product repeats are compared separately.'})
require_matplotlib_panel_alignment(fig,axes=eligible,panel_ids=ids,row_groups=rg,column_groups=cg,
    exemptions=ex,require_panel_labels=False,json_out=QA/'alignment.json',overlay_svg=QA/'alignment.svg',strict=True)
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.png',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.pdf',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.svg',dpi=450,facecolor='white')
fig.canvas.draw()
meta=[{'axis':j,'bounds':list(a.get_position().bounds),'title':a.get_title(),'xlabel':a.get_xlabel(),'ylabel':a.get_ylabel(),
       'texts':[t.get_text() for t in a.texts]} for j,a in enumerate(fig.axes)]
(QA/'axes.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'Figure {FIGURE_NUMBER}: exported, {len(before)} numerical artist arrays unchanged.')
plt.close(fig)
